# Trabalho Final — Aprendizado Profundo
## Classificação de grandes peixes migratórios com CNN, ResNet50 e ConvNeXtTiny

**Tema:** desenvolvimento de modelos de aprendizagem profunda para identificação de grandes peixes migratórios capturados pela pesca oceânica brasileira.

### Objetivo experimental
Comparar, sob o **mesmo split estratificado** e a mesma resolução de entrada:

1. **E1 — CNN própria treinada do zero** — baseline;
2. **E2 — ResNet50** — transfer learning + fine-tuning;
3. **E3 — ConvNeXtTiny** — transfer learning + fine-tuning.

O notebook cobre: análise exploratória, divisão treino/validação/teste, pré-processamento, augmentation, balanceamento por `class_weight`, histórico de treinamento, Accuracy, Precision, Recall, F1, Top-3, matriz de confusão, ROC-AUC/PR-AUC multiclasses, custo computacional, comparação pareada e Grad-CAM.

> **Estado desta cópia:** EDA e os resultados históricos reais do ConvNeXt já existentes no Google Drive foram incorporados. O experimento comparativo final E1/E2/E3 foi preparado para execução no Google Colab, usando o dataset canônico do Drive. Os resultados da ResNet50 **não foram inventados**.

> **Saídas finais:** ao concluir os três experimentos, o notebook grava `comparacao_final.csv`, `comparacao_estatistica.csv`, `custo_computacional.csv`, matrizes de confusão e `RUN_COMPLETE.json` em `/content/drive/MyDrive/modelos_peixes/trabalho_final_dl`.


## 1. Desenho experimental

A comparação final deve ser feita sobre **exatamente as mesmas amostras**. Por isso, este notebook cria um split determinístico **70% treino / 15% validação / 15% teste**, estratificado por espécie, a partir do `metadata_imagens.csv`.

- O conjunto de **treino** ajusta pesos.
- A **validação** é usada por `EarlyStopping`, seleção de hiperparâmetros e checkpoint.
- O **teste** fica intocado até a avaliação final.
- `SEED = 123` é usado para permitir reprodução.
- Todas as redes recebem imagens RGB de `224 × 224`.
- As redes pré-treinadas usam pesos `ImageNet`.
- O backbone é treinado em duas etapas:
  1. **Transfer learning:** backbone congelado, treina apenas a cabeça.
  2. **Fine-tuning:** libera aproximadamente os 30% finais do backbone com learning rate baixo.

Isso atende ao requisito de baseline + pelo menos duas variações e evita uma comparação injusta baseada em splits diferentes.


In [ ]:
# =========================================================
# 2. SETUP — COLAB + REPRODUTIBILIDADE
# =========================================================
from pathlib import Path
import os, json, random, time, shutil, platform, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 123
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
TRAIN_FRAC = 0.70
VAL_FRAC = 0.15
TEST_FRAC = 0.15

random.seed(SEED)
np.random.seed(SEED)

try:
    import tensorflow as tf
    tf.random.set_seed(SEED)
    print("TensorFlow:", tf.__version__)
    print("GPUs:", tf.config.list_physical_devices("GPU"))
except Exception as e:
    raise RuntimeError(
        "Este notebook foi preparado para Google Colab com TensorFlow. "
        "Ative um runtime com GPU antes de continuar."
    ) from e

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    precision_recall_fscore_support, roc_auc_score, average_precision_score
)
from scipy.stats import binomtest

print("Python:", sys.version.split()[0])
print("Plataforma:", platform.platform())


In [ ]:
# =========================================================
# 3. GOOGLE DRIVE — SOMENTE LEITURA DO DATASET CANÔNICO
# =========================================================
from google.colab import drive
drive.mount("/content/drive")

CANONICAL_ROOT = Path(
    "/content/drive/MyDrive/"
    "dataset_peixes_inaturalist_completo/"
    "dataset_peixes_inaturalist"
)
IMAGE_ROOT = CANONICAL_ROOT / "images"
METADATA_FILE = CANONICAL_ROOT / "metadata_imagens.csv"

MODEL_DIR = Path("/content/drive/MyDrive/modelos_peixes/trabalho_final_dl")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

assert CANONICAL_ROOT.exists(), f"Raiz canônica não encontrada: {CANONICAL_ROOT}"
assert IMAGE_ROOT.exists(), f"Pasta de imagens não encontrada: {IMAGE_ROOT}"
assert METADATA_FILE.exists(), f"Metadata não encontrado: {METADATA_FILE}"

print("Dataset:", CANONICAL_ROOT)
print("Saídas:", MODEL_DIR)


In [1]:
# =========================================================
# 4. EDA — METADATA E DISTRIBUIÇÃO DAS CLASSES
# =========================================================
metadata = pd.read_csv(METADATA_FILE)
required_cols = {"especie", "arquivo_local", "fonte", "licenca"}
missing = required_cols - set(metadata.columns)
if missing:
    raise ValueError(f"Colunas obrigatórias ausentes: {sorted(missing)}")

metadata["path"] = metadata["arquivo_local"].apply(lambda p: str(CANONICAL_ROOT / p))
metadata["exists"] = metadata["path"].apply(os.path.exists)

print("Registros no metadata:", len(metadata))
print("Arquivos existentes:", int(metadata["exists"].sum()))
if not metadata["exists"].all():
    display(metadata.loc[~metadata["exists"], ["especie", "arquivo_local"]].head(20))
    raise RuntimeError("Há linhas de metadata sem arquivo físico. Corrija antes de treinar.")

class_counts = metadata["especie"].value_counts().sort_values(ascending=False)
source_counts = metadata["fonte"].value_counts()

display(pd.DataFrame({"imagens": class_counts}))
display(pd.DataFrame({"imagens": source_counts}))

fig, ax = plt.subplots(figsize=(10, 5))
class_counts.sort_values().plot(kind="barh", ax=ax)
ax.set_title(f"Distribuição por espécie (n={len(metadata)})")
ax.set_xlabel("Número de imagens")
ax.set_ylabel("")
for i, v in enumerate(class_counts.sort_values().values):
    ax.text(v + 5, i, str(v), va="center")
plt.tight_layout()
plt.show()


Registros no metadata: 3652
Distribuição registrada no metadata canônico:
especie
Coryphaena hippurus        500
Istiophorus platypterus    499
Kajikia albida              30
Katsuwonus pelamis         760
Makaira nigricans          384
Tetrapturus pfluegeri       29
Thunnus alalunga           335
Thunnus albacares          500
Thunnus obesus             115
Xiphias gladius            500

Origem das imagens:
fonte
iNaturalist          2856
GBIF                  673
Wikimedia Commons     123


### Diagnóstico do dataset

O dataset contém **3.652 registros** no metadata canônico e é claramente desbalanceado. A maior classe tem 760 imagens e a menor 29, uma razão de aproximadamente **26:1**. Isso torna **Accuracy isoladamente insuficiente**; por isso, a comparação inclui F1 macro, Precision/Recall macro e métricas por classe.

Além disso, há diversidade de proveniência (iNaturalist, GBIF e Wikimedia Commons), o que é positivo para variabilidade visual, mas exige rastreabilidade de licença e origem.


In [ ]:
# =========================================================
# 5. SPLIT 70/15/15 — ESTRATIFICADO E REPRODUTÍVEL
# =========================================================
df_train, df_temp = train_test_split(
    metadata,
    test_size=(VAL_FRAC + TEST_FRAC),
    random_state=SEED,
    stratify=metadata["especie"],
)

relative_test = TEST_FRAC / (VAL_FRAC + TEST_FRAC)
df_val, df_test = train_test_split(
    df_temp,
    test_size=relative_test,
    random_state=SEED,
    stratify=df_temp["especie"],
)

for name, frame in [("train", df_train), ("validation", df_val), ("test", df_test)]:
    print(f"{name:10s}: {len(frame):4d} imagens")
    display(frame["especie"].value_counts().sort_index().rename("n").to_frame())

split_manifest = pd.concat([
    df_train.assign(split="train"),
    df_val.assign(split="validation"),
    df_test.assign(split="test"),
], ignore_index=True)

manifest_cols = [c for c in [
    "especie", "arquivo_local", "fonte", "licenca", "observation_id",
    "photo_id", "source_record_id", "split"
] if c in split_manifest.columns]

split_manifest[manifest_cols].to_csv(
    MODEL_DIR / "split_manifest_70_15_15.csv",
    index=False
)
print("Manifest salvo em:", MODEL_DIR / "split_manifest_70_15_15.csv")


In [ ]:
# =========================================================
# 6. TF.DATA — MESMAS AMOSTRAS PARA TODOS OS MODELOS
# =========================================================
class_names = sorted(metadata["especie"].unique().tolist())
class_to_id = {name: i for i, name in enumerate(class_names)}
NUM_CLASSES = len(class_names)

def make_dataset(frame, training=False):
    paths = frame["path"].astype(str).values
    labels = frame["especie"].map(class_to_id).astype("int32").values
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    def load_image(path, label):
        raw = tf.io.read_file(path)
        img = tf.io.decode_image(raw, channels=3, expand_animations=False)
        img.set_shape([None, None, 3])
        img = tf.image.resize(img, IMG_SIZE, antialias=True)
        img = tf.cast(img, tf.float32)
        return img, label

    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(df_train, training=True)
val_ds = make_dataset(df_val)
test_ds = make_dataset(df_test)

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_CLASSES),
    y=df_train["especie"].map(class_to_id).values
)
class_weight = {i: float(w) for i, w in enumerate(weights)}

print("Classes:")
for i, name in enumerate(class_names):
    print(f"{i:2d} -> {name:28s} | class_weight={class_weight[i]:.3f}")


In [ ]:
# =========================================================
# 7. DATA AUGMENTATION
# =========================================================
from tensorflow.keras import layers

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.10),
    layers.RandomContrast(0.10),
], name="data_augmentation")

images, labels = next(iter(train_ds))
aug = data_augmentation(images[:9], training=True)

plt.figure(figsize=(10, 8))
for i in range(min(9, len(aug))):
    plt.subplot(3, 3, i + 1)
    plt.imshow(tf.cast(tf.clip_by_value(aug[i], 0, 255), tf.uint8))
    plt.title(class_names[int(labels[i])].replace(" ", "\n"), fontsize=8)
    plt.axis("off")
plt.suptitle("Exemplo de data augmentation")
plt.tight_layout()
plt.show()


In [ ]:
# =========================================================
# 8. E1 — CNN PRÓPRIA (BASELINE)
# =========================================================
from tensorflow.keras import Model, Input

def build_baseline_cnn():
    inp = Input(shape=(*IMG_SIZE, 3), name="imagem")
    x = data_augmentation(inp)
    x = layers.Rescaling(1./255)(x)

    for filters in [32, 64, 128]:
        x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(256, 3, padding="same", activation="relu")(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.35)(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax", name="classificacao")(x)

    model = Model(inp, out, name="cnn_baseline")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3_accuracy"),
        ],
    )
    return model

baseline_model = build_baseline_cnn()
baseline_model.summary()


In [ ]:
# =========================================================
# 9. E2/E3 — BACKBONES PRÉ-TREINADOS
# =========================================================
def build_transfer_model(backbone_name):
    inp = Input(shape=(*IMG_SIZE, 3), name="imagem")
    x = data_augmentation(inp)

    if backbone_name == "ResNet50":
        base = tf.keras.applications.ResNet50(
            include_top=False,
            weights="imagenet",
            input_shape=(*IMG_SIZE, 3),
        )
        x = tf.keras.applications.resnet.preprocess_input(x)
        dropout = 0.30

    elif backbone_name == "ConvNeXtTiny":
        base = tf.keras.applications.ConvNeXtTiny(
            include_top=False,
            include_preprocessing=True,
            weights="imagenet",
            input_shape=(*IMG_SIZE, 3),
        )
        dropout = 0.20
    else:
        raise ValueError(backbone_name)

    base.trainable = False
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D(name="global_average_pooling")(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax", name="classificacao")(x)

    return Model(inp, out, name=f"{backbone_name}_peixes"), base

def compile_stage1(model):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3_accuracy"),
        ],
    )

def enable_fine_tuning(model, base, unfreeze_fraction=0.30):
    base.trainable = True
    cut = int(len(base.layers) * (1 - unfreeze_fraction))
    for layer in base.layers[:cut]:
        layer.trainable = False
    for layer in base.layers[cut:]:
        layer.trainable = True

    # Para ResNet, não atualizar estatísticas de BatchNorm com dataset pequeno.
    for layer in base.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False

    model.compile(
        optimizer=tf.keras.optimizers.AdamW(
            learning_rate=1e-5, weight_decay=1e-5, clipnorm=1.0
        ),
        loss="sparse_categorical_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3_accuracy"),
        ],
    )
    print("Camadas do backbone:", len(base.layers))
    print("Camadas treináveis:", sum(int(l.trainable) for l in base.layers))


In [ ]:
# =========================================================
# 10. TREINAMENTO PADRONIZADO
# =========================================================
def callbacks_for(name):
    return [
        tf.keras.callbacks.ModelCheckpoint(
            MODEL_DIR / f"{name}_best.keras",
            monitor="val_loss", save_best_only=True, verbose=1,
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=5, restore_best_weights=True, verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.2, patience=2, min_lr=1e-7, verbose=1,
        ),
        tf.keras.callbacks.CSVLogger(MODEL_DIR / f"{name}_history.csv"),
    ]

def fit_baseline():
    model = build_baseline_cnn()
    start = time.perf_counter()
    hist = model.fit(
        train_ds, validation_data=val_ds, epochs=30,
        class_weight=class_weight,
        callbacks=callbacks_for("E1_CNN_baseline"), verbose=1,
    )
    return model, hist, time.perf_counter() - start

def fit_transfer(backbone_name):
    model, base = build_transfer_model(backbone_name)
    compile_stage1(model)
    t0 = time.perf_counter()

    h1 = model.fit(
        train_ds, validation_data=val_ds, epochs=30,
        class_weight=class_weight,
        callbacks=callbacks_for(f"{backbone_name}_stage1"), verbose=1,
    )

    enable_fine_tuning(model, base, unfreeze_fraction=0.30)

    h2 = model.fit(
        train_ds, validation_data=val_ds, epochs=20,
        class_weight=class_weight,
        callbacks=callbacks_for(f"{backbone_name}_finetuned"), verbose=1,
    )

    seconds = time.perf_counter() - t0
    model.save(MODEL_DIR / f"{backbone_name}_final.keras")
    return model, h1, h2, seconds


In [ ]:
# =========================================================
# 11. EXECUTAR OS 3 EXPERIMENTOS
# =========================================================
EXECUTE_TRAINING = True
artifacts = {}

if EXECUTE_TRAINING:
    print("\n=== E1: CNN baseline ===")
    baseline_model, baseline_hist, baseline_time = fit_baseline()
    artifacts["CNN baseline"] = {"model": baseline_model, "time": baseline_time}

    print("\n=== E2: ResNet50 ===")
    resnet_model, resnet_h1, resnet_h2, resnet_time = fit_transfer("ResNet50")
    artifacts["ResNet50"] = {"model": resnet_model, "time": resnet_time}

    print("\n=== E3: ConvNeXtTiny ===")
    convnext_model, convnext_h1, convnext_h2, convnext_time = fit_transfer("ConvNeXtTiny")
    artifacts["ConvNeXtTiny"] = {"model": convnext_model, "time": convnext_time}
else:
    print("Treinamento desativado.")


In [ ]:
# =========================================================
# 12. AVALIAÇÃO FINAL — SOMENTE TESTE
# =========================================================
def collect_predictions(model, ds):
    y_true, probs = [], []
    for xb, yb in ds:
        pb = model.predict(xb, verbose=0)
        y_true.extend(yb.numpy().tolist())
        probs.append(pb)
    return np.asarray(y_true), np.concatenate(probs, axis=0)

def evaluate_model(name, model, train_seconds):
    y_true, probs = collect_predictions(model, test_ds)
    y_pred = probs.argmax(axis=1)

    acc = accuracy_score(y_true, y_pred)
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    _, _, f1_weight, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )

    y_onehot = tf.one_hot(y_true, depth=NUM_CLASSES).numpy()
    try:
        roc_macro = roc_auc_score(y_onehot, probs, multi_class="ovr", average="macro")
    except ValueError:
        roc_macro = np.nan
    try:
        pr_macro = average_precision_score(y_onehot, probs, average="macro")
    except ValueError:
        pr_macro = np.nan

    top3 = np.mean([
        int(y_true[i] in np.argpartition(probs[i], -3)[-3:])
        for i in range(len(y_true))
    ])

    metrics = {
        "model": name,
        "test_accuracy": float(acc),
        "macro_precision": float(p_macro),
        "macro_recall": float(r_macro),
        "macro_f1": float(f1_macro),
        "weighted_f1": float(f1_weight),
        "top3_accuracy": float(top3),
        "roc_auc_macro_ovr": float(roc_macro),
        "pr_auc_macro": float(pr_macro),
        "parameters": int(model.count_params()),
        "training_seconds": float(train_seconds),
    }

    (MODEL_DIR / f"{name.replace(' ', '_')}_evaluation.json").write_text(
        json.dumps(metrics, indent=2), encoding="utf-8"
    )

    print(f"\n===== {name} =====")
    print(json.dumps(metrics, indent=2))
    print(classification_report(
        y_true, y_pred,
        labels=np.arange(NUM_CLASSES),
        target_names=class_names,
        digits=4, zero_division=0,
    ))

    cm = confusion_matrix(y_true, y_pred, labels=np.arange(NUM_CLASSES))
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(cm)
    ax.set_title(f"Matriz de confusão — {name}")
    ax.set_xlabel("Previsto")
    ax.set_ylabel("Real")
    ax.set_xticks(range(NUM_CLASSES), [x.replace(" ", "\n") for x in class_names], rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(NUM_CLASSES), class_names, fontsize=8)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=7)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    fig.savefig(MODEL_DIR / f"{name.replace(' ', '_')}_confusion_matrix.png", dpi=180)
    plt.show()
    return metrics, y_true, y_pred, probs

results, predictions = [], {}
if EXECUTE_TRAINING:
    for name, item in artifacts.items():
        metrics, y_true, y_pred, probs = evaluate_model(name, item["model"], item["time"])
        results.append(metrics)
        predictions[name] = (y_true, y_pred, probs)

    results_df = pd.DataFrame(results).sort_values("test_accuracy", ascending=False)
    display(results_df)
    results_df.to_csv(MODEL_DIR / "comparacao_final.csv", index=False)


In [ ]:
# =========================================================
# 13. COMPARAÇÃO PAREADA — McNEMAR EXATO + BOOTSTRAP
# =========================================================
def mcnemar_exact(y_true, pred_a, pred_b):
    a_ok = pred_a == y_true
    b_ok = pred_b == y_true
    n10 = int(np.sum(a_ok & ~b_ok))
    n01 = int(np.sum(~a_ok & b_ok))
    n = n10 + n01
    p = 1.0 if n == 0 else binomtest(min(n10, n01), n=n, p=0.5, alternative="two-sided").pvalue
    return n10, n01, p

def bootstrap_accuracy_diff(y_true, pred_a, pred_b, n_boot=5000, seed=SEED):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    diffs = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        diffs[i] = np.mean(pred_a[idx] == y_true[idx]) - np.mean(pred_b[idx] == y_true[idx])
    return np.quantile(diffs, [0.025, 0.5, 0.975])

if EXECUTE_TRAINING:
    rows = []
    for a, b in [
        ("ResNet50", "CNN baseline"),
        ("ConvNeXtTiny", "CNN baseline"),
        ("ConvNeXtTiny", "ResNet50"),
    ]:
        y_true, pred_a, _ = predictions[a]
        _, pred_b, _ = predictions[b]
        n10, n01, p = mcnemar_exact(y_true, pred_a, pred_b)
        ci = bootstrap_accuracy_diff(y_true, pred_a, pred_b)
        rows.append({
            "modelo_A": a, "modelo_B": b,
            "A_acerta_B_erra": n10, "A_erra_B_acerta": n01,
            "mcnemar_p": p,
            "acc_diff_mediana": ci[1],
            "acc_diff_IC95_inf": ci[0],
            "acc_diff_IC95_sup": ci[2],
        })
    stats_df = pd.DataFrame(rows)
    display(stats_df)
    stats_df.to_csv(MODEL_DIR / "comparacao_estatistica.csv", index=False)


In [ ]:
# =========================================================
# 14. CUSTO COMPUTACIONAL E LATÊNCIA
# =========================================================
def benchmark_latency(model, ds, warmup=3, repetitions=20):
    xb, _ = next(iter(ds))
    xb = xb[:1]
    for _ in range(warmup):
        _ = model(xb, training=False)

    times = []
    for _ in range(repetitions):
        t0 = time.perf_counter()
        _ = model(xb, training=False)
        times.append((time.perf_counter() - t0) * 1000.0)

    return {"latency_ms_mean": float(np.mean(times)),
            "latency_ms_std": float(np.std(times))}

if EXECUTE_TRAINING:
    complexity_rows = []
    for name, item in artifacts.items():
        b = benchmark_latency(item["model"], test_ds)
        complexity_rows.append({
            "model": name,
            "parameters": item["model"].count_params(),
            "training_minutes": item["time"] / 60.0,
            **b,
        })
    complexity_df = pd.DataFrame(complexity_rows)
    display(complexity_df)
    complexity_df.to_csv(MODEL_DIR / "custo_computacional.csv", index=False)


## 15. Explicabilidade com Grad-CAM

Grad-CAM ajuda a verificar **onde** a CNN concentrou evidência para uma decisão. No trabalho escrito, ele deve ser tratado como análise qualitativa complementar — não como prova causal de que o modelo "entendeu" o peixe.


In [ ]:
# =========================================================
# 15. GRAD-CAM GENÉRICO
# =========================================================
def find_last_4d_layer(model):
    candidates = []
    def visit(layer):
        try:
            if len(layer.output.shape) == 4:
                candidates.append(layer)
        except Exception:
            pass
        if hasattr(layer, "layers"):
            for sub in layer.layers:
                visit(sub)
    visit(model)
    if not candidates:
        raise RuntimeError("Nenhuma camada 4-D encontrada.")
    return candidates[-1]

def gradcam_heatmap(model, image_batch, class_index=None):
    conv_layer = find_last_4d_layer(model)
    grad_model = tf.keras.Model(model.inputs, [conv_layer.output, model.output])

    with tf.GradientTape() as tape:
        conv_outputs, preds = grad_model(image_batch, training=False)
        if class_index is None:
            class_index = tf.argmax(preds[0])
        score = preds[:, class_index]

    grads = tape.gradient(score, conv_outputs)
    weights = tf.reduce_mean(grads, axis=(1, 2))
    cam = tf.reduce_sum(conv_outputs * weights[:, None, None, :], axis=-1)
    cam = tf.nn.relu(cam)
    cam = cam / (tf.reduce_max(cam, axis=(1, 2), keepdims=True) + 1e-8)
    return cam[0].numpy(), int(class_index), preds[0].numpy()

if EXECUTE_TRAINING:
    xb, yb = next(iter(test_ds))
    model = artifacts["ConvNeXtTiny"]["model"]
    heatmap, pred_idx, prob = gradcam_heatmap(model, xb[:1])

    base_img = xb[0].numpy().astype("uint8")
    hm = tf.image.resize(heatmap[..., None], IMG_SIZE).numpy()[..., 0]

    plt.figure(figsize=(10,4))
    plt.subplot(1,2,1)
    plt.imshow(base_img)
    plt.title(f"Real: {class_names[int(yb[0])]}")
    plt.axis("off")

    plt.subplot(1,2,2)
    plt.imshow(base_img)
    plt.imshow(hm, cmap="jet", alpha=0.45)
    plt.title(f"Previsto: {class_names[pred_idx]}\nconf={prob[pred_idx]:.1%}")
    plt.axis("off")
    plt.tight_layout()
    gradcam_path = MODEL_DIR / "ConvNeXtTiny_gradcam_example.png"
    plt.savefig(gradcam_path, dpi=180, bbox_inches="tight")
    plt.show()
    print("Grad-CAM salvo em:", gradcam_path)


## 16. Resultado piloto já executado — ConvNeXtTiny

Antes desta padronização 70/15/15, já havia um experimento ConvNeXtTiny executado no projeto com `224×224`, `batch=32`, backbone ImageNet, transfer learning e fine-tuning.

**Resultados reais registrados:**

| Métrica | ConvNeXtTiny |
|---|---:|
| Accuracy | **0,8192** |
| Top-3 Accuracy | **0,9493** |
| Loss | **0,5896** |
| F1 macro | **0,7549** |
| F1 ponderado | **0,8180** |
| Amostras de validação | **730** |

A CNN inicial treinada do zero havia registrado `val_accuracy = 0,4972`, servindo como evidência preliminar de ganho substancial com transfer learning/fine-tuning.

> **Importante:** esses números são históricos e usam o split anterior 80/20. Eles são apresentados como **resultado piloto**, não devem ser usados como a comparação final contra ResNet50. A tabela final deve vir do bloco E1/E2/E3 acima, que mantém o mesmo split de teste para todos os modelos.


In [2]:
# Resultado piloto real, importado do histórico existente no projeto
pilot = pd.DataFrame([
    {"modelo": "CNN do zero (histórico)", "accuracy": 0.4972, "observacao": "validação histórica"},
    {"modelo": "ConvNeXtTiny TL+FT (histórico)", "accuracy": 0.8192, "observacao": "validação histórica"},
])
display(pilot)

ax = pilot.plot(x="modelo", y="accuracy", kind="bar",
                ylim=(0,1), legend=False, figsize=(7,4), rot=0)
ax.set_ylabel("Accuracy")
ax.set_title("Resultado piloto já executado")
for p in ax.patches:
    ax.annotate(f"{p.get_height():.1%}",
                (p.get_x()+p.get_width()/2, p.get_height()),
                ha="center", va="bottom", xytext=(0,4),
                textcoords="offset points")
plt.tight_layout()
plt.show()


                        modelo  accuracy          observacao
       CNN do zero (histórico)    0.4972 validação histórica
ConvNeXtTiny TL+FT (histórico)    0.8192 validação histórica

## 17. Como interpretar os resultados

- **Accuracy:** proporção total de acertos.
- **F1 macro:** dá o mesmo peso a cada espécie; é central neste dataset desbalanceado.
- **F1 ponderado:** reflete mais as classes frequentes.
- **Top-3:** útil quando espécies visualmente próximas aparecem entre as principais hipóteses.
- **Matriz de confusão:** identifica pares de espécies confundidos.
- **ROC-AUC macro OVR / PR-AUC macro:** complementam a análise probabilística.
- **McNemar:** testa as discordâncias de acerto/erro nas mesmas imagens.
- **Bootstrap:** fornece IC95% para a diferença de acurácia.
- **Parâmetros, tempo e latência:** quantificam custo computacional.

### Hipóteses
**H1:** transfer learning supera a CNN treinada do zero.  
**H2:** fine-tuning melhora generalização em relação à cabeça com backbone congelado.  
**H3:** ConvNeXtTiny e ResNet50 exibem perfis de erro diferentes.  
**H4:** classes raras tendem a apresentar Recall/F1 inferiores, mesmo com `class_weight`.


In [ ]:
# =========================================================
# 18. EXPORTAÇÃO PARA RELATÓRIO/PPT
# =========================================================
print("Pasta de resultados:", MODEL_DIR)
for p in sorted(MODEL_DIR.glob("*")):
    print("-", p.name)


## Checklist final da disciplina

- [x] Problema claramente definido  
- [x] Dataset próprio documentado  
- [x] EDA e desbalanceamento analisados  
- [x] Split treino/validação/teste  
- [x] Pré-processamento e augmentation  
- [x] Baseline CNN  
- [x] ResNet50 com transfer learning + fine-tuning  
- [x] ConvNeXtTiny com transfer learning + fine-tuning  
- [x] Histórico de treinamento e checkpoints  
- [x] Accuracy, Precision, Recall, F1, Top-3, matriz de confusão  
- [x] ROC-AUC / PR-AUC multiclasses  
- [x] Comparação estatística pareada  
- [x] Custo computacional  
- [x] Grad-CAM  
- [x] Reprodutibilidade (`SEED`, manifest do split, versões)  

**Antes de entregar:** execute o notebook de cima a baixo no Colab, confirme que as três linhas aparecem em `comparacao_final.csv` e use apenas os resultados do mesmo split no relatório e na apresentação.


In [ ]:
# =========================================================
# 16. MARCADOR DE CONCLUSÃO — PARA ATUALIZAÇÃO DO RELATÓRIO
# =========================================================
# Esta célula é executada somente depois que todos os experimentos terminarem.
# Ela cria um pequeno arquivo no Drive para indicar que os resultados finais
# já estão disponíveis para leitura e atualização do DOCX/PDF/PPT.

from datetime import datetime, timezone

if EXECUTE_TRAINING:
    expected_outputs = [
        "comparacao_final.csv",
        "comparacao_estatistica.csv",
        "custo_computacional.csv",
        "CNN_baseline_confusion_matrix.png",
        "ResNet50_confusion_matrix.png",
        "ConvNeXtTiny_confusion_matrix.png",
    ]

    missing = [name for name in expected_outputs if not (MODEL_DIR / name).exists()]
    if missing:
        print("ATENÇÃO: alguns arquivos esperados não foram encontrados:", missing)
    else:
        marker = {
            "status": "complete",
            "finished_at_utc": datetime.now(timezone.utc).isoformat(),
            "model_dir": str(MODEL_DIR),
            "results_file": "comparacao_final.csv",
            "statistics_file": "comparacao_estatistica.csv",
            "cost_file": "custo_computacional.csv",
            "models": ["CNN baseline", "ResNet50", "ConvNeXtTiny"],
            "seed": SEED,
            "split": "70/15/15",
        }
        marker_path = MODEL_DIR / "RUN_COMPLETE.json"
        marker_path.write_text(json.dumps(marker, indent=2), encoding="utf-8")
        print("✅ Experimento final concluído.")
        print("Marcador salvo em:", marker_path)
        print("Agora os resultados podem ser usados para atualizar o relatório e a apresentação.")
